# 00 - Setup and configuration

**Stage question.** How is a single experimental run identified, and can that identifier be trusted to be stable where it must be and sensitive where it must be?

Every fit in CODA is keyed by a hash of its configuration. Results are written under that key and never overwritten. Two properties have to hold, and both are load-bearing for claims made much later:

1. **Stability.** Two runs of the same fit must produce the same identifier, even when one ran on this machine and the other on a cloud runtime. Otherwise the same analysis reproduced elsewhere looks like a different experiment.
2. **Sensitivity.** Two runs differing in *anything* that affects the result must produce different identifiers. Otherwise two experimental conditions could silently write into the same place, and the second would overwrite the first without warning.

This notebook exercises `src/config.py` directly. It defines no logic of its own.

**Terminology used below.**

- **Condition** - one of the four information settings the main experiment compares. They differ only in what the model is allowed to see.
- **Donor scheme** - a control in which the partner's turn is replaced by a different turn, to test whether a result depends on the genuine pairing.
- **Partner mode** - how the partner's acoustic pathway is fed. Changing it changes the information available and never the number of parameters.
- **Block** - a group of related delivery measurements. Statistical testing happens at block level, not per individual measurement.

In [1]:
import json
import sys
from dataclasses import replace
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    TARGET_BLOCKS, TARGET_DIMENSIONS, N_TARGET_DIMENSIONS,
    RunConfig, DataConfig, NeuralConfig,
    environment_record, assert_same_capacity,
)

## 1. The target being predicted

The quantity this study predicts is the *delivery* of a reply: how it was said, not what was said. It is measured as eight numbers per turn, grouped into four blocks.

In [2]:
for block, dims in TARGET_BLOCKS.items():
    print(f'{block:<20} {len(dims)}  ' + ', '.join(dims))
print()
print(f'{N_TARGET_DIMENSIONS} dimensions in {len(TARGET_BLOCKS)} blocks')

pitch_level_span     2  mean_log_f0, log_f0_range_p95_p5
pitch_boundary       1  log_f0_slope_final_300ms
energy               2  mean_intensity, intensity_range
timing               3  articulation_rate, pause_ratio, longest_internal_pause

8 dimensions in 4 blocks


**What this shows.** Eight measurements in four groups: where the voice sits in pitch and how far it ranges; how the pitch moves at the very end of the turn; how loud it is and how much the loudness varies; and three aspects of timing.

**Why it is grouped.** Statistical testing is done per block, with each block weighted equally, giving four tests rather than eight. This matters for two reasons. Testing eight correlated measurements separately would inflate the chance of a spurious finding. And weighting by block stops timing, which has three measurements, from dominating the pitch boundary, which has one, purely by count.

**A caution worth recording.** Earlier project drafts describe this target as *nine* dimensions in their prose while listing eight in the table. No draft names a ninth. The enumeration is the definition, and the correction is recorded in the preregistration before any confirmatory result. The relevant consequence is narrow: it changes how wide the prediction head is, and nothing about the hypotheses, the tests, or the controls.

## 2. Run identifiers

A configuration describes one fit completely. Its hash is that fit's name.

In [3]:
primary = RunConfig(experiment='j1', condition='plus_partner_delivery')
print('primary comparison :', primary.run_id())
print('results directory  :', primary.result_dir())

primary comparison : 9621e0cd70b458dc
results directory  : results\j1__9621e0cd70b458dc


### 2.1 Stability

The same configuration must always resolve to the same identifier, and free-text notes must not disturb it.

In [4]:
same = RunConfig(experiment='j1', condition='plus_partner_delivery')
noted = replace(primary, notes='rerun after a preempted cloud session')

print('rebuilt identically :', same.run_id() == primary.run_id())
print('notes added         :', noted.run_id() == primary.run_id())

rebuilt identically : True
notes added         : True


**What this shows.** Rebuilding the configuration from scratch reproduces the identifier, and annotating a run does not change it.

**Why it matters.** Part of this project's computation runs on a cloud runtime and part runs locally. If the environment leaked into the identifier, the same fit would be filed under two names and its provenance would fracture. The environment is therefore *recorded* alongside each run but deliberately excluded from the hash.

### 2.2 Sensitivity

Anything that could change the result must change the identifier. The four experimental conditions are the case that matters most.

In [5]:
conds = ['Z_content', 'plus_partner_text', 'plus_partner_delivery', 'plus_self_previous']
for c in conds:
    print(f'{c:<24} ' + replace(primary, condition=c).run_id())
print()
print('all distinct:', len({replace(primary, condition=c).run_id() for c in conds}) == 4)

Z_content                806cc72561bd60e7
plus_partner_text        cecea9047b3521a6
plus_partner_delivery    9621e0cd70b458dc
plus_self_previous       4557f90467770603

all distinct: True


In [6]:
# A change buried inside a nested configuration must still surface.
deeper = replace(primary, data=DataConfig(split_seed=7))
print('changed split seed ->', deeper.run_id(),
      '| differs:', deeper.run_id() != primary.run_id())

changed split seed -> c056159d4dde63bb | differs: True


**What this shows.** Each condition gets its own identifier, and a change buried several levels down in the configuration still produces a different one.

**Why it matters.** The central comparison in this study is between conditions that are deliberately similar, differing only in what information the model may use. They will be run repeatedly, under several control schemes and several random seeds. A collision between any two would mean one silently overwrote another's results, and the error would be invisible in the output. The sensitivity check is what rules that out.

**What this does not establish.** That the hash is cryptographically meaningful, or that the configuration captures everything relevant. It captures what has been declared. A change made outside the configuration, by editing code, is not visible here, which is why the commit is recorded separately.

## 3. Equal capacity across conditions

The neural part of this study compares conditions that see different information. For that comparison to say anything about *information*, everything else must be held fixed: the same architecture, the same number of trainable parameters, the same optimiser and training budget.

This is the single easiest way for a study like this to fool itself. A model given more input and more capacity will usually do better, and the improvement says nothing about whether the extra input carried useful information. So capacity equality is asserted in code rather than assumed.

In [7]:
shared = NeuralConfig()
modes = ('real', 'donor', 'null_token', 'delivery_vector_interpretable')
runs = [RunConfig(experiment='neural', condition='plus_partner_delivery',
                  partner_mode=m, neural=shared) for m in modes]

assert_same_capacity(runs)
print('capacity identical across all four partner modes')
for r in runs:
    print(f'  {r.partner_mode:<32} ' + r.run_id())

capacity identical across all four partner modes
  real                             d195fe34bc8477ba
  donor                            3313acf2963cb733
  null_token                       6bfa2f5884d9f8e6
  delivery_vector_interpretable    a2256133d6b24aee


In [8]:
# And it must fail loudly when an architecture actually differs.
tampered = [runs[0], replace(runs[1], neural=NeuralConfig(d_model=256))]
try:
    assert_same_capacity(tampered)
    print('NOT CAUGHT - this would be a defect')
except AssertionError as err:
    print('caught as expected:', err)

caught as expected: Neural configuration differs across conditions, which breaks capacity control: neural/plus_partner_delivery


**What this shows.** Four different ways of feeding the partner's audio share one architecture and one parameter count, yet each is filed under its own identifier. When a genuine architectural difference is introduced, the check refuses it.

**Why it matters.** The comparison this study relies on is *same model, same capacity, same training, different information available*. Published work in this area often compares systems that differ in modality, size and optimisation all at once, which is why those results establish that acoustic context helps without establishing how much it contributes. The assertion above is what keeps this study out of that trap.

**Note on the ablation.** In the condition where the partner's audio is withheld, the pathway is not deleted. It receives a learned placeholder sequence instead. Deleting it would shrink the model and reintroduce the very confound being controlled for.

## 4. Execution environment

Work is split between this machine and a cloud runtime. Each run records where it happened.

In [9]:
print(json.dumps(environment_record(), indent=2))

{
  "runtime": "local",
  "git_commit": "93b7432f90441b29f6930ef6804a2e61954c332c",
  "python": "3.13.15",
  "platform": "Windows-10-10.0.19045-SP0"
}


**What this shows.** The runtime, the commit the code was run from, and the interpreter version.

**Why it matters.** A result produced in the cloud has to be reproducible from this checkout, and that only holds if both environments are pinned to the same library versions. The commit recorded here is what links a cloud artifact back to the exact code that produced it. The pitch-tracking library is the sharpest case: if its defaults differed between the two environments, every target value in the study would shift, and nothing downstream would announce the problem.

A `None` commit means the code was run outside a repository checkout. That is acceptable for exploration and not acceptable for a recorded run.

## 5. Where results go

One directory per run, named by experiment and identifier.

In [10]:
for c in conds:
    print(replace(primary, condition=c).result_dir())

results\j1__806cc72561bd60e7
results\j1__cecea9047b3521a6
results\j1__9621e0cd70b458dc
results\j1__4557f90467770603


**What this shows.** Each condition writes to its own directory, so results accumulate rather than overwrite.

**Why it matters.** Every number that eventually appears in a figure or table has to be traceable to the run that produced it. Keeping run directories separate and append-only is the mechanism. It also means an experiment can be re-run after a change without destroying the earlier evidence, which is what allows a change of approach to be documented rather than silently erased.

## 6. Summary

The run-identifier scheme behaves as required: stable across environments and across incidental annotation, sensitive to every declared change including nested ones, and sufficient to keep the four experimental conditions separate on disk.

Capacity equality across the partner modes is asserted in code and fails loudly when violated.

**Limitations of what was checked here.** Only *declared* configuration is hashed. Changes made by editing code are not visible to it, which is why the commit is recorded alongside each run and why library versions are pinned rather than left to resolve. Neither check has yet been exercised against a cloud runtime; that happens at the first extraction session, and agreement between the two environments is verified separately on a fixed reference sample before any bulk processing runs.

**Next stage.** Corpus audit and channel quality, which produces the first decision gate.